## Tujuan Preprocessing
Notebook ini menyiapkan data final untuk modeling.
Tahapan:
1. Data loading & overview
2. Cleaning dasar (drop tenure=0, konversi tipe data)
3. Feature engineering (membuat fitur turunan)
4. Encoding kategori
5. Train-test split

In [1]:
# Import library
import pandas as pd
import numpy as np
import joblib, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load dataset
df = pd.read_csv("../dataset/raw/Telco-Customer-Churn.csv")
print(f"Dataset shape sebelum preprocessing: {df.shape}")
display(df.head())
print(f"\nInfo dataset:")
df.info()

Dataset shape sebelum preprocessing: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



Info dataset:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str 

## Data Overview
Mengecek struktur awal data sebelum transformasi

In [3]:
# Cleaning dasar
df.columns = df.columns.str.strip()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print(f"Baris dengan tenure=0: {(df['tenure'] == 0).sum()}")
df = df[df['tenure'] > 0].reset_index(drop=True)

print(f"\nDataset shape setelah drop tenure=0: {df.shape}")
print(f"Missing TotalCharges: {df['TotalCharges'].isna().sum()}")

Baris dengan tenure=0: 11

Dataset shape setelah drop tenure=0: (7032, 21)
Missing TotalCharges: 0


In [4]:
# Validasi cleaning
print("Missing values setelah cleaning:")
print(df.isnull().sum().sort_values(ascending=False))

# Cek duplikat
print(f"\nJumlah duplikat: {df.duplicated().sum()}")

# Cek tipe data
print("\nTipe data numerik:")
display(df.select_dtypes(include=['number']).dtypes)

Missing values setelah cleaning:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Jumlah duplikat: 0

Tipe data numerik:


SeniorCitizen       int64
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object

## Target dan Fitur
- Target: Churn (Yes/No)
- Fitur: kolom lain yang akan diproses untuk modeling

In [5]:
# Pemisahan Fitur (X) dan Target (y)
y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["customerID", "Churn"])

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"\nProporsi churn:")
print(y.value_counts(normalize=True).mul(100).round(2))

X shape: (7032, 19), y shape: (7032,)

Proporsi churn:
Churn
0    73.42
1    26.58
Name: proportion, dtype: float64


## Feature Engineering
Membuat fitur turunan yang lebih informatif untuk modeling

In [6]:
# 1. Average Monthly Charges
X["AvgMonthlyCharges"] = X["TotalCharges"] / X["tenure"]

# 2. Tenure Bucket
def tenure_bucket(tenure):
    if tenure <= 12:
        return "New" # 0-1 tahun
    elif tenure <= 24:
        return "Early" # 1-2 tahun
    elif tenure <=48:
        return "Established" # 2-4 tahun
    else:
        return "Loyal" # > 4 tahun

X["TenureBucket"] = X["tenure"].apply(tenure_bucket)

# 3. Total Services
service_cols = ["PhoneService", "OnlineSecurity",
                "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
X["TotalServices"] = X[service_cols].apply(lambda row: (row == "Yes").sum(), axis=1)

print("Feature engineering selesai. Fitur baru:")
print(X[["AvgMonthlyCharges", "TenureBucket", "TotalServices"]].head(10))

Feature engineering selesai. Fitur baru:
   AvgMonthlyCharges TenureBucket  TotalServices
0          29.850000          New              1
1          55.573529  Established              3
2          54.075000          New              3
3          40.905556  Established              3
4          75.825000          New              1
5         102.562500          New              4
6          88.609091        Early              3
7          30.190000          New              1
8         108.787500  Established              5
9          56.257258        Loyal              3


In [7]:
# Validasi feature engineering
print("Statistik AvgMonthlyCharges:")
print(X["AvgMonthlyCharges"].describe())

print(f"\nDistribusi TenureBucket:")
print(X["TenureBucket"].value_counts())

print(f"\nDistribusi TotalServices:")
print(X["TotalServices"].value_counts().sort_index())

print(f"\nInf di AvgMonthlyCharges: {np.isinf(X['AvgMonthlyCharges']).sum()}")
print(f"NaN di AvgMonthlyCharges: {X['AvgMonthlyCharges'].isna().sum()}")

Statistik AvgMonthlyCharges:
count    7032.000000
mean       64.799424
std        30.185891
min        13.775000
25%        36.179891
50%        70.373239
75%        90.179560
max       121.400000
Name: AvgMonthlyCharges, dtype: float64

Distribusi TenureBucket:
TenureBucket
Loyal          2239
New            2175
Established    1594
Early          1024
Name: count, dtype: int64

Distribusi TotalServices:
TotalServices
0      80
1    2247
2     996
3    1041
4    1060
5     825
6     524
7     259
Name: count, dtype: int64

Inf di AvgMonthlyCharges: 0
NaN di AvgMonthlyCharges: 0


In [8]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Encoding Kategori
Mengubah kolom kategorikal menjadi numerik agar model bisa memproses

In [9]:
# Binary encoding
binary_cols = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling"
]
for col in binary_cols:
    X_train[col] = X_train[col].map({"Yes": 1, "No": 0})
    X_test[col] = X_test[col].map({"Yes": 1, "No": 0})

# One-hot encoding
categorical_cols = [
    "gender", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract", "PaymentMethod", "TenureBucket"
]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True).astype(int)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True).astype(int)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

drop_cols = ["TenureBucket"]

print(f"Binary columns: {binary_cols}")
print(f"Categorical columns: {categorical_cols}")

Binary columns: ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Categorical columns: ['gender', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod', 'TenureBucket']


## Scaling Fitur Numerik
Normalize fitur numerik untuk model yang sensitif terhadap skala
(Optional untuk tree-based model, tapi diperlukan untuk linear model)

In [10]:
# scale hanya kolom numerik tertentu
scale_cols = ["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlyCharges", "TotalServices"]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [11]:
print("Shape final:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Shape final:
X_train: (5625, 35)
X_test : (1407, 35)


In [12]:
assert X_train.select_dtypes(include=["object"]).shape[1] == 0, "X_train masih punya kolom object"
assert X_test.select_dtypes(include=["object"]).shape[1] == 0, "X_test masih punya kolom object"
assert set(X_train.columns) == set(X_test.columns), "Kolom train-test belum selaras"
print("Final check passed: data siap modeling")

Final check passed: data siap modeling


## Output Final Preprocessing
Data sudah siap untuk modeling.
Disimpan dalam variabel:
- X_train, X_test: fitur untuk training dan testing
- y_train, y_test: target untuk training dan testing

In [13]:
# Simpan dataset untuk dipakai ulang di modeling
X_train.to_csv("../dataset/processed/X_train.csv", index=False)
X_test.to_csv("../dataset/processed/X_test.csv", index=False)
y_train.to_csv("../dataset/processed/y_train.csv", index=False)
y_test.to_csv("../dataset/processed/y_test.csv", index=False)

print("Dataset disimpan di folder dataset/processed/")
print(f"X_train.csv: {X_train.shape}")
print(f"X_test.csv: {X_test.shape}")
print(f"y_train.csv: {y_train.shape}")
print(f"y_test.csv: {y_test.shape}")

Dataset disimpan di folder dataset/processed/
X_train.csv: (5625, 35)
X_test.csv: (1407, 35)
y_train.csv: (5625,)
y_test.csv: (1407,)


In [14]:
preprocessor_artifact = {
    "expected_columns": X_train.columns.tolist(),
    "scale_cols": scale_cols,
    "scaler": scaler
}
os.makedirs("../models", exist_ok=True)
joblib.dump(preprocessor_artifact, "../models/preprocessor.joblib")
print("Saved preprocessor to ../models/preprocessor.joblib")

Saved preprocessor to ../models/preprocessor.joblib
